# Scene 2 Experimental OpenAI Realization Notebook

In [ ]:
%cd /content
!git clone https://github.com/UrbinaDan/PaperTheater_ai_Pipeline.git
%cd PaperTheater_ai_Pipeline

!pip install -q numpy scipy matplotlib opencv-python-headless pillow shapely svgwrite cairosvg tqdm networkx pyyaml requests openai accelerate bitsandbytes einops sentencepiece transformers==4.49.0


In [ ]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI key: ")


In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
!wget -q -P checkpoints https://raw.githubusercontent.com/facebookresearch/sam2/main/sam2_configs/sam2_hiera_s.yaml

%cd /content
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd Depth-Anything-V2
!mkdir -p checkpoints
!wget -q -P checkpoints https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth


In [ ]:
%cd /content/PaperTheater_ai_Pipeline

import importlib, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.occlusion_heuristic import heuristic_complete
from src.mask_cleanup import cleanup_mask

from src.scene_builder import parse_florence_boxes, merge_segmented_by_label, build_stable_merged_objects
from src.scene_representation import build_scene_representation
from src.layer_planner import plan_layers_deterministic
from src.layer_renderer import build_object_mask_map
from src.layer_context_builder import build_layer_contexts
from src.layer_realization_openai import realize_single_layer_experimental

import src.occlusion_openai
importlib.reload(src.occlusion_openai)
openai_edit = src.occlusion_openai.openai_edit

paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)


In [ ]:
SCENE_NAME = "scene_2"
SOURCE_IMAGE_PATH = "/mnt/data/scene_2.jpg"
TARGET_IMAGE_PATH = f"/content/PaperTheater_ai_Pipeline/data/input/{SCENE_NAME}.jpg"

Path(TARGET_IMAGE_PATH).parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE_IMAGE_PATH, TARGET_IMAGE_PATH)

image = load_image(TARGET_IMAGE_PATH, max_side=cfg.image_max_side)

plt.imshow(image)
plt.axis("off")
plt.title("Scene 2")
plt.show()


In [ ]:
florence = FlorenceParser(cfg.florence_model)
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
depth_runner = DepthRunner(cfg.depth_encoder)

caption = florence.get_dense_caption(image)
print(caption)

det = florence.get_open_vocab_detection(
    image,
    "a person, deer, trees, branches, bushes, grass, path, building, sky, clouds"
)


In [ ]:
boxes = parse_florence_boxes(det, image.shape)
segmented = segmenter.segment_boxes(image, boxes)
merged_segmented = merge_segmented_by_label(segmented)

depth = depth_runner.infer(image)
stable_objects = build_stable_merged_objects(merged_segmented, depth)


In [ ]:
scene_repr = build_scene_representation(
    image_path=TARGET_IMAGE_PATH,
    image_shape=image.shape,
    caption=caption,
    stable_objects=stable_objects,
)

layer_plan = plan_layers_deterministic(scene_repr)


In [ ]:
openai_results = []

for obj in stable_objects:
    mask = obj["mask"]
    label = obj["label"]
    cleaned_mask = cleanup_mask(heuristic_complete(mask, label), cfg.min_component_area, cfg.smooth_kernel)

    out_mask = paths.completed_dir / f"{SCENE_NAME}_mask_{obj['id']}.png"
    save_mask(out_mask, cleaned_mask)

    openai_results.append({
        "id": obj["id"],
        "label": label,
        "bbox": obj["bbox"],
        "mask_path": str(out_mask)
    })


In [ ]:
object_mask_map = build_object_mask_map(openai_results)

layer_contexts = build_layer_contexts(
    scene_repr=scene_repr,
    layer_plan=layer_plan,
    object_mask_map=object_mask_map,
)


In [ ]:
def apply_focus_mask(img, own, front):
    out = img.copy()
    out[~(own>0)] = 0
    out[(front>0)] = 0
    return out

def experimental_layer_openai_fn(image_crop, ownership_mask_crop, front_occlusion_mask_crop, layer_context, prompt, model_name):
    focused = apply_focus_mask(image_crop, ownership_mask_crop, front_occlusion_mask_crop)
    return openai_edit(focused, ownership_mask_crop, layer_context["labels"][0], model_name, prompt=prompt)


In [ ]:
all_results = []

for ctx in layer_contexts:
    label = ctx["labels"][0] if ctx["labels"] else ctx["layer_name"]

    prompt = f"Generate only the {label} layer. No extra objects. Respect mask."

    result = realize_single_layer_experimental(
        image=image,
        layer_context=ctx,
        prompt=prompt,
        output_dir="/content/experimental_scene2",
        openai_realize_fn=experimental_layer_openai_fn,
        model_name=cfg.openai_model,
        pad=8,
    )

    all_results.append(result)
    print(result["generated_visible_crop_path"])


In [ ]:
for r in all_results:
    img = Image.open(r["generated_visible_crop_path"])
    plt.imshow(img)
    plt.title(r["layer_name"])
    plt.axis("off")
    plt.show()
